In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00 - Setup & Sample Data Generation
# MAGIC
# MAGIC Creates the Unity Catalog catalog/schema/volumes used by the pipeline, and drops a few
# MAGIC batches of raw transaction files (CSV + JSON) into a "landing" volume to simulate a
# MAGIC client dropping files on a regular basis.
# MAGIC
# MAGIC Run this notebook once to set everything up, then run it again (or use the "drop next
# MAGIC batch" command at the bottom) to simulate new files arriving and test incremental
# MAGIC processing.

# COMMAND ----------

CATALOG = "lakehouse_demo"
SCHEMA = "transactions"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.schemas")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.quarantine_exports")

LANDING_CSV = f"/Volumes/{CATALOG}/{SCHEMA}/landing/csv"
LANDING_JSON = f"/Volumes/{CATALOG}/{SCHEMA}/landing/json"

import os
os.makedirs(LANDING_CSV, exist_ok=True)
os.makedirs(LANDING_JSON, exist_ok=True)

print("Catalog/schema/volumes ready.")
print("CSV landing zone: ", LANDING_CSV)
print("JSON landing zone:", LANDING_JSON)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data generator
# MAGIC
# MAGIC Generates realistic-but-messy transaction records:
# MAGIC - Batch 1 (CSV): "day 1" load
# MAGIC - Batch 2 (JSON): "day 2" load, includes a handful of transaction_ids that overlap with
# MAGIC   batch 1 (simulates re-sent/duplicate records) plus some null/invalid rows
# MAGIC - Batch 3 (JSON): "day 3" load, introduces a NEW column (`discount_pct`) that did not
# MAGIC   exist in earlier files, to exercise Auto Loader schema evolution

# COMMAND ----------

import csv
import json
import random
import uuid
from datetime import datetime, timedelta

random.seed(42)

CUSTOMERS = [f"CUST{str(i).zfill(4)}" for i in range(1, 26)]
CATEGORIES = ["Electronics", "Groceries", "Clothing", "Home", "Sports", "Books"]
PAYMENT_METHODS = ["card", "cash", "wallet", "bank_transfer"]
STATUSES = ["completed", "completed", "completed", "refunded", "failed"]  # weighted
CURRENCIES = ["GBP", "USD", "EUR"]


def make_record(txn_id, ts, dirty=False, with_discount=False):
    rec = {
        "transaction_id": txn_id,
        "customer_id": random.choice(CUSTOMERS),
        "transaction_ts": ts.strftime("%Y-%m-%dT%H:%M:%S"),
        "amount": round(random.uniform(5, 500), 2),
        "currency": random.choice(CURRENCIES),
        "category": random.choice(CATEGORIES),
        "payment_method": random.choice(PAYMENT_METHODS),
        "store_id": f"STORE{random.randint(1, 8):02d}",
        "status": random.choice(STATUSES),
    }
    if with_discount:
        rec["discount_pct"] = random.choice([0, 0, 5, 10, 15, 20])

    if dirty:
        # inject a data-quality problem
        problem = random.choice(["null_customer", "null_amount", "bad_currency", "negative_amount"])
        if problem == "null_customer":
            rec["customer_id"] = None
        elif problem == "null_amount":
            rec["amount"] = None
        elif problem == "bad_currency":
            rec["currency"] = "??"
        elif problem == "negative_amount":
            rec["amount"] = -abs(rec["amount"])
    return rec


def write_csv(path, records, fieldnames):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            writer.writerow(r)


def write_json_lines(path, records):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


def generate_batch_1():
    """CSV batch: day-1 load, 200 clean-ish rows, ~4% dirty."""
    base_ts = datetime(2026, 6, 1, 8, 0, 0)
    records = []
    for i in range(200):
        txn_id = f"TXN{100000 + i}"
        ts = base_ts + timedelta(minutes=i * 3)
        dirty = random.random() < 0.04
        records.append(make_record(txn_id, ts, dirty=dirty))
    fieldnames = ["transaction_id", "customer_id", "transaction_ts", "amount",
                  "currency", "category", "payment_method", "store_id", "status"]
    out_path = f"{LANDING_CSV}/transactions_batch_001.csv"
    write_csv(out_path, records, fieldnames)
    print("Wrote", out_path, "-", len(records), "rows")


def generate_batch_2():
    """JSON batch: day-2 load, 150 rows, includes 10 duplicate IDs from batch 1."""
    base_ts = datetime(2026, 6, 2, 8, 0, 0)
    records = []
    for i in range(140):
        txn_id = f"TXN{200000 + i}"
        ts = base_ts + timedelta(minutes=i * 3)
        dirty = random.random() < 0.05
        records.append(make_record(txn_id, ts, dirty=dirty))
    # re-send 10 records that already exist in batch 1 (simulates upstream re-delivery)
    for i in range(10):
        txn_id = f"TXN{100000 + i}"
        ts = base_ts + timedelta(minutes=(140 + i) * 3)
        records.append(make_record(txn_id, ts))
    out_path = f"{LANDING_JSON}/transactions_batch_002.json"
    write_json_lines(out_path, records)
    print("Wrote", out_path, "-", len(records), "rows (10 are duplicates of batch 1)")


def generate_batch_3():
    """JSON batch: day-3 load, 150 rows, introduces new `discount_pct` column (schema evolution)."""
    base_ts = datetime(2026, 6, 3, 8, 0, 0)
    records = []
    for i in range(150):
        txn_id = f"TXN{300000 + i}"
        ts = base_ts + timedelta(minutes=i * 3)
        dirty = random.random() < 0.04
        records.append(make_record(txn_id, ts, dirty=dirty, with_discount=True))
    out_path = f"{LANDING_JSON}/transactions_batch_003.json"
    write_json_lines(out_path, records)
    print("Wrote", out_path, "-", len(records), "rows (includes new discount_pct column)")


# COMMAND ----------

# MAGIC %md
# MAGIC Run these one at a time (rather than all at once) if you want to demonstrate the
# MAGIC pipeline picking up *new* files incrementally between runs. Running all three now is
# MAGIC fine too - Auto Loader will just pick up all of them in the first run.

# COMMAND ----------

generate_batch_1()

# COMMAND ----------

generate_batch_2()

# COMMAND ----------

generate_batch_3()

# COMMAND ----------

# MAGIC %md
# MAGIC ## To simulate a *later* incremental drop
# MAGIC Re-run this cell any time to drop one more file and then re-run `01_bronze_ingestion`
# MAGIC to see only the new file get picked up.

# COMMAND ----------

def generate_incremental_drop(batch_num=4):
    base_ts = datetime(2026, 6, 3 + batch_num - 3, 8, 0, 0)
    records = []
    for i in range(50):
        txn_id = f"TXN{(300000 + batch_num * 100000) + i}"
        ts = base_ts + timedelta(minutes=i * 3)
        records.append(make_record(txn_id, ts, dirty=random.random() < 0.04, with_discount=True))
    out_path = f"{LANDING_JSON}/transactions_batch_{batch_num:03d}.json"
    write_json_lines(out_path, records)
    print("Wrote", out_path, "-", len(records), "rows")

# generate_incremental_drop(4)
